# maplib + Polars: Your DataFrame is already a Knowledge Graph

CSV in, knowledge graph, SPARQL out.

## Create sample data

In [2]:
import polars as pl
from maplib import Model

df = pl.DataFrame({
    "sensor_id": ["S-001","S-002","S-003","S-004","S-005",
                  "S-006","S-007","S-008","S-009","S-010"],
    "location":  ["CompressorA","CompressorA","PumpB","PumpB","TurbineC",
                  "TurbineC","HeatExD","HeatExD","ValveE","ValveE"],
    "reading":   ["temperature","pressure","temperature","vibration","temperature",
                  "pressure","temperature","flow_rate","temperature","pressure"],
    "value":     [72.3, 14.7, 68.1, 0.42, 91.5,
                  22.1, 55.8, 120.3, 44.2, 11.9],
})
df

sensor_id,location,reading,value
str,str,str,f64
"""S-001""","""CompressorA""","""temperature""",72.3
"""S-002""","""CompressorA""","""pressure""",14.7
"""S-003""","""PumpB""","""temperature""",68.1
"""S-004""","""PumpB""","""vibration""",0.42
"""S-005""","""TurbineC""","""temperature""",91.5
"""S-006""","""TurbineC""","""pressure""",22.1
"""S-007""","""HeatExD""","""temperature""",55.8
"""S-008""","""HeatExD""","""flow_rate""",120.3
"""S-009""","""ValveE""","""temperature""",44.2


## Define the OTTR template

One template = the entire mapping from rows to triples.

In [3]:
m = Model()

m.add_template("""
    @prefix ex:  <http://example.com/plant/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:SensorReading [
        ottr:IRI   ?sensor,
        ottr:IRI   ?location,
        xsd:string ?reading,
        xsd:double ?value
    ] :: {
        ottr:Triple(?sensor,   a,              ex:Sensor),
        ottr:Triple(?sensor,   rdfs:label,     ?reading),
        ottr:Triple(?sensor,   ex:locatedAt,   ?location),
        ottr:Triple(?sensor,   ex:hasValue,    ?value),
        ottr:Triple(?location, a,              ex:Equipment)
    } .
""")

## Map the DataFrame to knowledge graph

In [4]:
ex = "http://example.com/plant/"

m.map(ex + "SensorReading", df.select(
    sensor   = pl.lit(ex) + pl.col("sensor_id"),
    location = pl.lit(ex) + pl.col("location"),
    reading  = "reading",
    value    = "value",
))

print(f"{m.size()} triples from {df.shape[0]} rows")

45 triples from 10 rows


## SPARQL returns Polars DataFrame

Every query result comes back as a native Polars DataFrame.

### Readings per equipment

In [5]:
m.query("""
    PREFIX ex: <http://example.com/plant/>
    SELECT ?equipment (COUNT(?s) AS ?sensors) (AVG(?v) AS ?avg_reading)
    WHERE {
        ?s ex:locatedAt ?equipment ; ex:hasValue ?v .
    }
    GROUP BY ?equipment
    ORDER BY DESC(?avg_reading)
""")

equipment,sensors,avg_reading
str,u32,f64
"""<http://example.com/plant/Heat…",2,88.05
"""<http://example.com/plant/Turb…",2,56.8
"""<http://example.com/plant/Comp…",2,43.5
"""<http://example.com/plant/Pump…",2,34.26
"""<http://example.com/plant/Valv…",2,28.05


### Temperature sensors only

In [6]:
m.query("""
    PREFIX ex:   <http://example.com/plant/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?equipment ?temp
    WHERE {
        ?s ex:locatedAt ?equipment ;
           rdfs:label   "temperature" ;
           ex:hasValue  ?temp .
    }
    ORDER BY DESC(?temp)
""")

equipment,temp
str,f64
"""<http://example.com/plant/Turb…",91.5
"""<http://example.com/plant/Comp…",72.3
"""<http://example.com/plant/Pump…",68.1
"""<http://example.com/plant/Heat…",55.8
"""<http://example.com/plant/Valv…",44.2


### Equipment with above-average temperatures

Subquery computes the global average, outer query filters.

In [7]:
m.query("""
    PREFIX ex:   <http://example.com/plant/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?equipment ?temp
    WHERE {
        ?s ex:locatedAt ?equipment ;
           rdfs:label   "temperature" ;
           ex:hasValue  ?temp .
        {
            SELECT (AVG(?v) AS ?avg)
            WHERE { ?x rdfs:label "temperature" ; ex:hasValue ?v }
        }
        FILTER(?temp > ?avg)
    }
    ORDER BY DESC(?temp)
""")

equipment,temp
str,f64
"""<http://example.com/plant/Turb…",91.5
"""<http://example.com/plant/Comp…",72.3
"""<http://example.com/plant/Pump…",68.1


### Enrich the graph with SPARQL CONSTRUCT

Add new triples derived from existing data — no external tools needed.

In [8]:
m.insert("""
    PREFIX ex:   <http://example.com/plant/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    CONSTRUCT {
        ?equipment ex:hotspot true
    }
    WHERE {
        ?s ex:locatedAt ?equipment ;
           rdfs:label   "temperature" ;
           ex:hasValue  ?temp .
        FILTER(?temp > 80)
    }
""")

m.query("""
    PREFIX ex: <http://example.com/plant/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?equipment WHERE { ?equipment ex:hotspot true }
""")

equipment
str
"""<http://example.com/plant/Turb…"


## Export

In [9]:
m.write("plant_graph.ttl", format="turtle")
print(f"Exported {m.size()} triples to plant_graph.ttl")

Exported 46 triples to plant_graph.ttl
